# 06 — Final controlled comparison and model selection

## Objective

Compare all five matched experiments in one table and select the deployable model
using validation-seen IoU only. Test-seen and test-unseen results describe final
performance but never influence model selection.


In [1]:
from pathlib import Path
import json
import os
import sys

from IPython.display import Image, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "datasets").is_dir() and (path / "final_model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_ID = os.environ.get("FINAL_TRAINING_RUN_ID", "manual")
FRESH_TRAINING = True
RESULT_ROOT = PROJECT_ROOT / "training_results_corrected"
POINT_ROOT = RESULT_ROOT
print("Project:", PROJECT_ROOT)
print("Run ID:", RUN_ID)


Project: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation
Run ID: manual


## 1. Comparison and selection implementation

This section validates the five experiment summaries, constructs the complete
comparison table, applies the validation-only selection rule, copies the selected
UI checkpoint, writes the model registry, and saves the primary comparison plot.


In [2]:
import json
import os
import shutil

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

EXPERIMENTS = {'baseline_object_mask': None, 'fixed_uvd': None, 'query_gated_uvd': None, 'rotation_consistent': None, 'geometry_dropout': None}

def run_id():
    return os.environ.get("FINAL_TRAINING_RUN_ID", "manual")

def results_root():
    path = PROJECT_ROOT / "training_results_corrected"
    path.mkdir(parents=True, exist_ok=True)
    return path

def points_root():
    path = PROJECT_ROOT / "training_results_corrected"
    path.mkdir(parents=True, exist_ok=True)
    return path

def build_comparison() -> pd.DataFrame:
    result_base = results_root()
    point_base = points_root()
    deployment_dir = PROJECT_ROOT / "models" / "final_study"
    missing_summaries = [
        name for name in EXPERIMENTS
        if not (result_base / name / "summary.csv").is_file()
    ]
    missing_checkpoints = [
        name for name in EXPERIMENTS
        if not (point_base / name / "ui_model.pt").is_file()
    ]
    if missing_summaries or missing_checkpoints:
        raise FileNotFoundError(
            f"Missing corrected artifacts; summaries={missing_summaries}, "
            f"UI checkpoints={missing_checkpoints}"
        )
    frames = [pd.read_csv(result_base / name / "summary.csv") for name in EXPERIMENTS]
    comparison = pd.concat(frames, ignore_index=True)
    comparison.to_csv(result_base / "all_experiment_comparison.csv", index=False)
    validation = comparison.drop_duplicates("experiment").sort_values("validation_iou", ascending=False)
    validation_by_model = validation.set_index("experiment")
    selected = str(validation.iloc[0].experiment)
    selected_source = point_base / selected / "ui_model.pt"

    # Keep the global winner in the corrected training record.
    shutil.copy2(selected_source, point_base / "best_model.pt")

    # Deploy the best-epoch inference checkpoint from every method so the UI
    # can switch models, plus a stable alias for the global winner.
    deployment_dir.mkdir(parents=True, exist_ok=True)
    for name in EXPERIMENTS:
        shutil.copy2(point_base / name / "ui_model.pt", deployment_dir / f"{name}.pt")
    shutil.copy2(selected_source, deployment_dir / "best_model.pt")
    registry = {
        "run_id": run_id(),
        "selection_rule": "maximum validation_seen IoU; no test metric used for selection",
        "selected_model": selected,
        "selected_checkpoint": str((deployment_dir / "best_model.pt").relative_to(PROJECT_ROOT)),
        "models": {
            name: {
                "checkpoint": str((deployment_dir / f"{name}.pt").relative_to(PROJECT_ROOT)),
                "validation_iou": float(validation_by_model.loc[name, "validation_iou"]),
                "selected_epoch": int(validation_by_model.loc[name, "selected_epoch"]),
            }
            for name in EXPERIMENTS
        },
    }
    registry_text = json.dumps(registry, indent=2) + "\n"
    (point_base / "model_registry.json").write_text(registry_text)
    (deployment_dir / "model_registry.json").write_text(registry_text)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    validation.sort_values("validation_iou").plot.barh(
        x="experiment", y="validation_iou", legend=False, ax=axes[0], title="Validation checkpoint selection"
    )
    test = comparison.pivot(index="experiment", columns="split", values="iou")
    test.plot.bar(ax=axes[1], title="Final seen and unseen test IoU")
    axes[0].set_xlabel("Validation IoU")
    axes[1].set_ylabel("Mean IoU")
    axes[1].tick_params(axis="x", rotation=30)
    figure.tight_layout()
    figure.savefig(result_base / "final_model_comparison.png", dpi=180, bbox_inches="tight")
    plt.close(figure)
    print(comparison.to_string(index=False))
    print("Selected model:", selected)
    print("Selected corrected checkpoint:", point_base / "best_model.pt")
    print("Deployment directory:", deployment_dir)
    return comparison


## 2. Ranking, seen/unseen comparison, and deployment choice

The standardized ranking below reports validation IoU, both final test splits,
generalization gaps, changes relative to the object-mask baseline, and leakage.


In [3]:
comparison = build_comparison()
display(comparison.round(4))

validation = comparison.drop_duplicates("experiment").set_index("experiment")
iou = comparison.pivot(index="experiment", columns="split", values="iou")
dice = comparison.pivot(index="experiment", columns="split", values="dice")
leakage = comparison.pivot(index="experiment", columns="split", values="leakage")

ranking = pd.DataFrame(index=validation.index)
ranking["selected epoch"] = validation["selected_epoch"].astype(int)
ranking["validation IoU"] = validation["validation_iou"]
ranking["seen IoU"] = iou["test_seen"]
ranking["unseen IoU"] = iou["test_unseen"]
ranking["seen Dice"] = dice["test_seen"]
ranking["unseen Dice"] = dice["test_unseen"]
ranking["seen leakage"] = leakage["test_seen"]
ranking["unseen leakage"] = leakage["test_unseen"]
ranking["seen−unseen IoU gap"] = ranking["seen IoU"] - ranking["unseen IoU"]

baseline_seen = ranking.loc["baseline_object_mask", "seen IoU"]
baseline_unseen = ranking.loc["baseline_object_mask", "unseen IoU"]
ranking["Δ seen IoU vs baseline"] = ranking["seen IoU"] - baseline_seen
ranking["Δ unseen IoU vs baseline"] = ranking["unseen IoU"] - baseline_unseen
ranking = ranking.sort_values("validation IoU", ascending=False)
display(ranking.round(4))
ranking.to_csv(RESULT_ROOT / "model_ranking.csv")

figure, axes = plt.subplots(1, 3, figsize=(17, 5))
ranking.sort_values("validation IoU")["validation IoU"].plot.barh(
    ax=axes[0], color="#38bdf8"
)
axes[0].set(title="Validation-only selection", xlabel="Validation IoU", ylabel="")

ranking[["seen IoU", "unseen IoU"]].plot.bar(
    ax=axes[1], color=("#818cf8", "#22c55e")
)
axes[1].set(title="Final test IoU", xlabel="", ylabel="Mean IoU", ylim=(0, 1))
axes[1].tick_params(axis="x", rotation=35)

ranking[["Δ seen IoU vs baseline", "Δ unseen IoU vs baseline"]].plot.bar(
    ax=axes[2], color=("#f59e0b", "#ec4899")
)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set(title="Improvement over baseline", xlabel="", ylabel="Δ IoU")
axes[2].tick_params(axis="x", rotation=35)

for axis in axes:
    axis.grid(alpha=0.25)
figure.tight_layout()
dashboard_path = RESULT_ROOT / "submission_model_comparison.png"
figure.savefig(dashboard_path, dpi=180, bbox_inches="tight")
plt.show()

registry = json.loads((POINT_ROOT / "model_registry.json").read_text())
selected = registry["selected_model"]
print("Selection rule:", registry["selection_rule"])
print("Selected model:", selected)
print("Selected validation IoU:", round(float(ranking.loc[selected, "validation IoU"]), 4))
print("Selected checkpoint:", PROJECT_ROOT / registry["selected_checkpoint"])
print("Deployment registry:", PROJECT_ROOT / "models" / "final_study" / "model_registry.json")
print("Comparison dashboard:", dashboard_path)


          experiment       split  samples      iou     dice  leakage  selected_epoch  validation_iou  validation_dice
baseline_object_mask   test_seen     3371 0.295609 0.394790 0.207410              20        0.288043         0.382185
baseline_object_mask test_unseen     1586 0.224319 0.308510 0.189084              20        0.288043         0.382185
           fixed_uvd   test_seen     3371 0.301535 0.399992 0.184232              15        0.291171         0.386212
           fixed_uvd test_unseen     1586 0.249311 0.338153 0.151189              15        0.291171         0.386212
     query_gated_uvd   test_seen     3371 0.302836 0.401979 0.194254              15        0.290894         0.386235
     query_gated_uvd test_unseen     1586 0.245555 0.333814 0.156482              15        0.290894         0.386235
 rotation_consistent   test_seen     3371 0.302785 0.403011 0.174673              14        0.293719         0.390326
 rotation_consistent test_unseen     1586 0.271211 0.368

,experiment,split,samples,iou,dice,leakage,selected_epoch,validation_iou,validation_dice
0,baseline_object_mask,test_seen,3371,0.2956,0.3948,0.2074,20,0.2880,0.3822
1,baseline_object_mask,test_unseen,1586,0.2243,0.3085,0.1891,20,0.2880,0.3822
2,fixed_uvd,test_seen,3371,0.3015,0.4000,0.1842,15,0.2912,0.3862
3,fixed_uvd,test_unseen,1586,0.2493,0.3382,0.1512,15,0.2912,0.3862
4,query_gated_uvd,test_seen,3371,0.3028,0.4020,0.1943,15,0.2909,0.3862
5,query_gated_uvd,test_unseen,1586,0.2456,0.3338,0.1565,15,0.2909,0.3862
6,rotation_consistent,test_seen,3371,0.3028,0.4030,0.1747,14,0.2937,0.3903
7,rotation_consistent,test_unseen,1586,0.2712,0.3683,0.1536,14,0.2937,0.3903
8,geometry_dropout,test_seen,3371,0.2972,0.3985,0.2144,17,0.2878,0.3850
9,geometry_dropout,test_unseen,1586,0.2415,0.3326,0.1746,17,0.2878,0.3850


,selected epoch,validation IoU,seen IoU,unseen IoU,seen Dice,unseen Dice,seen leakage,unseen leakage,seen−unseen IoU gap,Δ seen IoU vs baseline,Δ unseen IoU vs baseline
experiment,,,,,,,,,,,
rotation_consistent,14,0.2937,0.3028,0.2712,0.4030,0.3683,0.1747,0.1536,0.0316,0.0072,0.0469
fixed_uvd,15,0.2912,0.3015,0.2493,0.4000,0.3382,0.1842,0.1512,0.0522,0.0059,0.0250
query_gated_uvd,15,0.2909,0.3028,0.2456,0.4020,0.3338,0.1943,0.1565,0.0573,0.0072,0.0212
baseline_object_mask,20,0.2880,0.2956,0.2243,0.3948,0.3085,0.2074,0.1891,0.0713,0.0000,0.0000
geometry_dropout,17,0.2878,0.2972,0.2415,0.3985,0.3326,0.2144,0.1746,0.0557,0.0016,0.0172


Selection rule: maximum validation_seen IoU; no test metric used for selection
Selected model: rotation_consistent
Selected validation IoU: 0.2937
Selected checkpoint: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/models/final_study/best_model.pt
Deployment registry: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/models/final_study/model_registry.json
Comparison dashboard: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/training_results_corrected/submission_model_comparison.png


## 3. Conclusion and reporting checklist

The first-ranked model is selected exclusively from validation performance. The
final report should discuss whether its gains are consistent across seen and
unseen classes, whether leakage remains controlled, and whether the improvement
is large enough to justify the added geometry or regularization.

Saved outputs: complete comparison CSV, ranked model CSV, validation/test figures,
selected `best_model.pt`, and `model_registry.json`. The five compact per-method
checkpoints, global-best alias, and registry are also copied to `models/final_study/`
for the local inference UI.